# **Атаки на текстовую атрибуцию, amnesic probing и атака на SAE-представление**

Практика к модулю [«Атаки на объяснения»](https://ai-interpretability.school/learn/22/0/0), уроки «Атаки на текстовую атрибуцию» и «Зонды и иллюзии разреженных автокодировщиков».

В уроке «Атаки на текстовую атрибуцию» две атаки, TextExplanationFooler (TEF; Ivankay и соавторы, ICLR 2022) и ExplainFooler (Sinha и соавторы, BlackboxNLP 2021), заменой нескольких слов синонимами меняли атрибуцию классификатора текста, а предсказанный класс оставался прежним. В уроке «Зонды и иллюзии разреженных автокодировщиков» вывод «в представлении нашлось направление — значит, модель им пользуется» проверяли вмешательством (amnesic probing, Elazar и соавторы, 2021) и атакой на входе (Li и соавторы, 2026). Здесь все четыре метода считаются своим кодом на открытых моделях:

- раздел 1 — классификатор тональности DistilBERT на SST-2 и атрибуция Integrated Gradients (IG) из `captum`;
- раздел 2 — насколько атрибуция меняется сама, без противника: опорный вход IG и число шагов;
- раздел 3 — ExplainFooler: порядок слов по leave-one-out, цель — L2-расстояние атрибуций, сравнение со случайным выбором синонима;
- раздел 4 — TEF: порядок слов по нулевому эмбеддингу, цель — корреляция Пирсона, площадь ACC и случайная атака RA;
- раздел 5 — amnesic probing: INLP на слое DistilBERT, удаление двух свойств и контроль случайными направлениями;
- раздел 6 — атака на SAE-представление GPT-2 small заменой одного токена.

**Почему не TextAttack.** В уроке «Атаки на текстовую атрибуцию» сказано, что поиск TEF повторяется на [TextAttack](https://github.com/QData/TextAttack), и ExplainFooler написан на нем же. Но TextAttack 0.3.11 при импорте тянет `flair`, а `flair` требует `transformers` 4.x и в Colab с `transformers` 5 ставится только с понижением версии. Поэтому три составные части атаки — кандидаты-синонимы, ограничения и жадный поиск — написаны здесь своим кодом по тем же правилам: синонимы из WordNet, как в `WordSwapWordNet` TextAttack, той же части речи, без стоп-слов, каждое слово заменяется не больше одного раза, предсказанный класс сохраняется. Counter-fitted эмбеддингов, из которых берут синонимы обе статьи, в WordNet нет, поэтому кандидатов меньше.

**Среда.** Тетрадь проверена на версиях Colab (`torch` 2.11, `transformers` 5.18); `captum`, `sae-lens` и `transformer-lens` ставятся первой ячейкой. Работает и на процессоре, на GPU быстрее. Случайность — только в выборе предложений и в случайных точках сравнения, сид фиксирован.

In [ ]:
%pip install -q captum==0.9.0 nltk==3.9.1 transformer-lens==3.9.0 sae-lens==6.54.0

In [ ]:
import warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import wordnet as wn
from scipy.stats import spearmanr, pearsonr
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import IntegratedGradients, Saliency

for pkg in ('wordnet', 'stopwords', 'averaged_perceptron_tagger_eng'):
    nltk.download(pkg, quiet=True)
STOP = set(nltk.corpus.stopwords.words('english'))
warnings.filterwarnings('ignore', message='.*encountered in matmul')   # ложные предупреждения numpy на некоторых процессорах

SEED = 0
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
tok = AutoTokenizer.from_pretrained(NAME)
clf = AutoModelForSequenceClassification.from_pretrained(NAME).to(device).eval()
emb_layer = clf.distilbert.embeddings.word_embeddings
sst = load_dataset('stanfordnlp/sst2', split='validation')
print('устройство:', device, '| предложений в валидационной части SST-2:', len(sst))

## 1. Классификатор и атрибуция IG по словам

`distilbert-base-uncased-finetuned-sst-2-english` — DistilBERT, дообученный на тональности SST-2; на этой же модели ExplainFooler считает таблицу 1. Атрибуцию считаем IG из `captum` по эмбеддингам токенов: функция `fwd` получает матрицу эмбеддингов и возвращает логит нужного класса, DistilBERT сам прибавляет к ней позиционные эмбеддинги. Опорный вход по умолчанию — эмбеддинги токена `[PAD]` на всех позициях, кроме `[CLS]` и `[SEP]`, 50 шагов: так у ExplainFooler. TEF берет опорным входом нулевую матрицу эмбеддингов, это `baseline='zero'`.

Слово может распасться на несколько подсловных токенов. Как у ExplainFooler, слово получает ту оценку своих токенов, у которой наибольший модуль; так векторы атрибуции до и после замены слова имеют одну длину — число слов.

Слова — это части предложения, разделенные пробелами: в SST-2 текст уже разбит на слова и знаки препинания.

**Задание 1. Посчитайте точность классификатора на всех 872 предложениях валидационной части. Затем для первого предложения из `sents` проверьте полноту IG на уровне токенов: сумма атрибуций всех токенов, включая `[CLS]` и `[SEP]`, должна быть близка к разности логитов на входе и на опорном входе.**

In [ ]:
@torch.no_grad()
def predict(sentences, bs=64):
    out = []
    for i in range(0, len(sentences), bs):
        enc = tok(sentences[i:i + bs], return_tensors='pt', padding=True).to(device)
        out.append(torch.softmax(clf(**enc).logits, -1).cpu())
    return torch.cat(out).numpy()


def fwd(emb, mask, target):
    return clf(inputs_embeds=emb, attention_mask=mask.expand(emb.shape[0], -1)).logits[:, target]


ig = IntegratedGradients(fwd)


def encode(words, baseline):
    enc = tok(words, is_split_into_words=True, return_tensors='pt').to(device)
    ids = enc['input_ids']
    with torch.no_grad():
        emb = emb_layer(ids)
        if baseline == 'pad':
            b_ids = torch.full_like(ids, tok.pad_token_id)
            b_ids[0, 0], b_ids[0, -1] = tok.cls_token_id, tok.sep_token_id
            base = emb_layer(b_ids)
        else:
            base = torch.zeros_like(emb)
    return enc, emb, base


def to_words(tok_attr, word_ids, n_words):
    # оценка слова — оценка его подсловного токена с наибольшим модулем
    out = np.zeros(n_words)
    for j, w in enumerate(word_ids):
        if w is not None and abs(tok_attr[j]) > abs(out[w]):
            out[w] = tok_attr[j]
    return out


def word_attr(words, target, n_steps=50, baseline='pad', zero_word=None):
    enc, emb, base = encode(words, baseline)
    wid = enc.word_ids()
    if zero_word is not None:          # TEF: эмбеддинг одного слова заменен нулевым
        emb = emb.clone()
        emb[0, [j for j, w in enumerate(wid) if w == zero_word]] = 0
    a = ig.attribute(emb, baselines=base, additional_forward_args=(enc['attention_mask'], target),
                     n_steps=n_steps, internal_batch_size=64)
    return to_words(a.sum(-1)[0].detach().cpu().numpy(), wid, len(words))


sents = [s for s in sst['sentence'] if 8 <= len(s.split()) <= 18]
rng = np.random.default_rng(SEED)
sample = [sents[i] for i in rng.choice(len(sents), 30, replace=False)]
w = sample[0].split()
label = int(predict([sample[0]])[0].argmax())
print('предложение:', sample[0], '| класс:', clf.config.id2label[label])
print({x: round(float(v), 3) for x, v in zip(w, word_attr(w, label))})

In [ ]:
# Ваш код здесь

## 2. Насколько атрибуция меняется без противника

Прежде чем мерить, насколько атака меняет атрибуцию, полезно знать, насколько она меняется от выборов самой исследовательницы. Сравним на 30 предложениях `sample` IG с опорным входом `[PAD]` и с нулевым опорным входом, а также IG за 20 и за 50 шагов против IG за 200 шагов. Мера — ранговая корреляция Спирмена векторов атрибуции по словам, как в таблице 1 ExplainFooler.

In [ ]:
rho = {'[PAD] vs zero baseline': [], '20 vs 200 steps': [], '50 vs 200 steps': []}
for s in sample:
    w = s.split()
    lab = int(predict([s])[0].argmax())
    a200 = word_attr(w, lab, 200)
    rho['[PAD] vs zero baseline'].append(spearmanr(word_attr(w, lab), word_attr(w, lab, baseline='zero')).statistic)
    rho['20 vs 200 steps'].append(spearmanr(word_attr(w, lab, 20), a200).statistic)
    rho['50 vs 200 steps'].append(spearmanr(word_attr(w, lab, 50), a200).statistic)
for k, v in rho.items():
    print(f'{k:>24}: средняя корреляция Спирмена {np.mean(v):.3f}, минимум {np.min(v):.3f}')

Число шагов почти не меняет ранги: 20 шагов достаточно, и дальше в поиске атак мы берем 20 шагов, чтобы он шел быстрее. Опорный вход меняет ранги сильно: у атрибуций с `[PAD]` и с нулевым эмбеддингом средняя корреляция Спирмена 0.432, это разные объяснения одного и того же прогноза. Это стоит помнить, читая числа ниже: атака, после которой корреляция с исходной атрибуцией падает до уровня строки «[PAD] vs zero baseline», меняет объяснение не больше, чем его меняет выбор опорного входа.

## 3. ExplainFooler: замены синонимами против IG

Кандидаты на замену — синонимы WordNet той же части речи; слово должно стоять в начальной форме (иначе синоним окажется в другой форме), стоп-слова и знаки препинания не трогаем. Жадный поиск идет по словам в заданном порядке; на каждом слове он перебирает синонимы, отбрасывает те, что меняют предсказанный класс, и из оставшихся берет лучший по цели атаки. Функция `attack` общая для двух атак, они различаются порядком слов и целью.

У ExplainFooler порядок задает leave-one-out: слово удаляют и смотрят, насколько падает вероятность предсказанного класса. Цель — L2-норма разности векторов атрибуции (L2 Norm, формула (3) статьи); точка сравнения — случайный синоним из прошедших ограничения. Доля замен растет до 0.5.

Ограничения на смысл предложения, которое в статьях проверяют с помощью Universal Sentence Encoder, здесь нет, и синонимы WordNet часто берутся из другого значения слова («credits» → «quotation»). Предсказание классификатора от этого не меняется по построению, но человек увидел бы такую замену. Это стоит помнить: атака без ограничения на смысл сильнее той, что в статьях.

In [ ]:
POS = {'NN': wn.NOUN, 'VB': wn.VERB, 'JJ': wn.ADJ, 'RB': wn.ADV}


def synonyms(words, k):
    cands = []
    for w, t in nltk.pos_tag(words):
        pos = POS[t[:2]] if t[:2] in POS else None
        if pos is None or w in STOP or not w.isalpha() or wn.morphy(w, pos) != w:
            cands.append([])
            continue
        syn = []
        for ss in wn.synsets(w, pos=pos):
            for lem in ss.lemma_names():
                lem = lem.lower()
                if lem.isalpha() and lem != w and lem not in syn:
                    syn.append(lem)
        cands.append(syn[:k])
    return cands


def attack(words, order, cands, objective, max_ratio, pick, baseline='pad', n_steps=20):
    """Жадная замена слов. Возвращает список шагов: (доля замененных слов, слова, атрибуция)."""
    label = int(predict([' '.join(words)])[0].argmax())
    a0 = word_attr(words, label, n_steps, baseline)
    cur, steps = list(words), [(0.0, list(words), a0)]
    for pos in order:
        if len(steps) / len(words) > max_ratio:
            break
        opts = [cur[:pos] + [c] + cur[pos + 1:] for c in cands[pos]]
        if not opts:
            continue
        keep = predict([' '.join(o) for o in opts]).argmax(1) == label     # класс сохраняется
        opts = [o for o, k in zip(opts, keep) if k]
        if not opts:
            continue
        if pick == 'random':
            cur = opts[rng.integers(len(opts))]
            a = word_attr(cur, label, n_steps, baseline)
        else:
            atts = [word_attr(o, label, n_steps, baseline) for o in opts]
            j = int(np.argmax([objective(a0, x) for x in atts]))
            cur, a = opts[j], atts[j]
        steps.append((len(steps) / len(words), list(cur), a))
    return steps


def l2(a0, a1):
    return np.linalg.norm(a0 - a1)          # L2 Norm of ExplainFooler, eq. (3): IG scores are not rescaled


def loo_order(words):
    label = int(predict([' '.join(words)])[0].argmax())
    p = predict([' '.join(words[:i] + words[i + 1:]) for i in range(len(words))])[:, label]
    return list(np.argsort(p))          # сначала слово, без которого уверенность падает сильнее всего


print(sample[1])
print({w: c[:5] for w, c in zip(sample[1].split(), synonyms(sample[1].split(), 50)) if c})

In [ ]:
ef = {'L2': [], 'random': []}
for s in sample:
    w = s.split()
    cands, order = synonyms(w, 50), loo_order(w)
    for arm in ef:
        steps = attack(w, order, cands, l2, 0.5, 'best' if arm == 'L2' else 'random')
        ef[arm].append([(r, max(spearmanr(steps[0][2], a).statistic, 0), ' '.join(x)) for r, x, a in steps])

BINS = [(0, 0.1), (0.1, 0.2), (0.2, 0.3), (0.3, 0.4), (0.4, 0.5)]


def by_bin(runs):
    # для каждого предложения берем последний шаг внутри интервала доли замен
    out = []
    for lo, hi in BINS:
        vals = [[c for r, c, _ in run if lo < r <= hi][-1] for run in runs if any(lo < r <= hi for r, _, _ in run)]
        out.append((np.mean(vals) if vals else np.nan, len(vals)))
    return out


for arm, runs in ef.items():
    print(arm, ' | '.join(f'{lo}-{hi}: {m:.3f} (n={n})' for (lo, hi), (m, n) in zip(BINS, by_bin(runs))))
best = min(range(len(sample)), key=lambda i: ef['L2'][i][-1][1])
print('\nдо атаки:   ', ef['L2'][best][0][2])
print('после атаки:', ef['L2'][best][-1][2], f'| Спирмен {ef["L2"][best][-1][1]:.3f}')

In [ ]:
centers = [(lo + hi) / 2 for lo, hi in BINS]
plt.figure(figsize=(5.5, 3.6))
for arm, color in (('L2', '#c0392b'), ('random', '#7f8c8d')):
    plt.plot(centers, [m for m, _ in by_bin(ef[arm])], 'o-', color=color, label=f'ExplainFooler, {arm} choice')
plt.axhline(np.mean(rho['[PAD] vs zero baseline']), ls='--', color='#3b6ea5', label='[PAD] vs zero baseline, no attack')
plt.xlabel('Share of replaced words')
plt.ylabel('Spearman with original IG (clipped at 0)')
plt.title('DistilBERT, SST-2, 30 sentences')
plt.ylim(0, 1)
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

Как и в таблице 1 ExplainFooler, выбор синонима по L2 снижает корреляцию с исходной атрибуцией сильнее, чем случайный синоним при той же доле замен, а класс при этом не меняется ни на одном шаге. Число предложений в правых интервалах меньше 30: у коротких предложений с малым числом заменимых слов поиск кончается раньше, чем доля замен доходит до 0.5.

## 4. TEF: нулевой эмбеддинг, корреляция Пирсона и случайная атака

У TEF два отличия от ExplainFooler. Порядок слов: эмбеддинг каждого слова по очереди заменяют нулевым и смотрят, насколько меняется атрибуция по той же мере $d$, что и в цели ниже, как в шаге 1 статьи; слова сортируют по убыванию изменения. Цель: $d=1-\frac{\mathrm{PCC}+1}{2}$, где PCC — корреляция Пирсона атрибуций исходного и атакованного текста. Синонимов $N=15$, доля замен до $\rho_{\max}=0.4$, опорный вход IG — нулевой. Точка сравнения RA: случайный порядок слов и случайный синоним с теми же фильтрами.

Сводная мера ACC — площадь под кривой средней PCC по доле замен $\rho$ от 0 до 0.4. Чтобы кривая была определена в каждой точке, после окончания поиска PCC предложения держится на последнем достигнутом значении.

**Задание 2. Запустите TEF и RA на 30 предложениях `sample`, постройте кривые средней PCC на сетке $\rho = 0, 0.05, \dots, 0.4$ и посчитайте ACC обеих атак методом трапеций.**

In [ ]:
def tef_order(words):
    label = int(predict([' '.join(words)])[0].argmax())
    a0 = word_attr(words, label, 20, 'zero')
    change = [pcc_objective(a0, word_attr(words, label, 20, 'zero', zero_word=i)) for i in range(len(words))]   # the same distance d as the objective
    return list(np.argsort(change)[::-1])


def pcc_objective(a0, a1):
    return 1 - (pearsonr(a0, a1).statistic + 1) / 2


GRID = np.round(np.arange(0, 0.401, 0.05), 2)


def curve(steps):
    r = np.array([x for x, _, _ in steps])
    pcc = np.array([pearsonr(steps[0][2], a).statistic for _, _, a in steps])
    return np.array([pcc[r <= g + 1e-9][-1] for g in GRID])

In [ ]:
# Ваш код здесь

ACC у TEF ниже, чем у RA: при одной и той же доле замен атрибуция TEF-текста дальше от исходной. В статье для BERT с IG на Yelp ACC у TEF вышла отрицательной (−0.14 против 0.10 у RA); у нас кандидатов-синонимов меньше и предложения короче, поэтому разрыв между TEF и RA меньше.

**Перенос на другой объяснитель.** Замены TEF искали против IG. Сохраняется ли эффект для Saliency — модуля градиента по эмбеддингу, суммированного по координатам? Посчитаем PCC атрибуций Saliency исходного и атакованного текста.

In [ ]:
sal = Saliency(fwd)


def word_saliency(words, target):
    enc, emb, _ = encode(words, 'zero')
    g = sal.attribute(emb, additional_forward_args=(enc['attention_mask'], target), abs=True)
    return to_words(g.sum(-1)[0].detach().cpu().numpy(), enc.word_ids(), len(words))


tr = []
for s0, s1 in tef_texts:
    if s0 == s1:
        continue
    lab = int(predict([s0])[0].argmax())
    tr.append(pearsonr(word_saliency(s0.split(), lab), word_saliency(s1.split(), lab)).statistic)
print(f'предложений, где TEF что-то заменил: {len(tr)}; средняя PCC Saliency до и после: {np.mean(tr):.3f}')

У Ivankay и соавторов перенос замен на другую пару «модель и объяснитель» тоже ослабевал: атака без переноса снижает PCC сильнее.

## 5. Amnesic probing: удалить свойство и посмотреть на поведение

Зонд показывает, что свойство извлекается из представления, но не то, что модель им пользуется. Elazar и соавторы проверяют использование вмешательством: свойство удаляют из представления INLP (Iterative Nullspace Projection) и смотрят, изменилось ли поведение модели.

INLP своим кодом: логистическая регрессия `sklearn` предсказывает свойство по усредненному по токенам скрытому состоянию слоя 5 DistilBERT; представления проецируют на нуль-пространство ее весов; повторяют, пока точность нового зонда на валидационной части не окажется в пределах одного пункта от точности самого частого класса, но не больше 40 итераций. Проекцию $P$ затем применяют ко всем позициям на выходе слоя 5 при прямом проходе классификатора. Контроль Rand: столько же случайных направлений.

Свойства два:

- **тональность** — то, что классификатор и должен предсказывать;
- **длина** — длиннее ли предложение медианы валидационной части (в словах): свойство, которое легко извлечь, но которое для тональности не нужно.

Обучающие представления — 4000 предложений обучающей части SST-2 длиной от 6 слов (обучающая часть SST-2 содержит и короткие фразы).

**Задание 3. Для каждого свойства выведите точность зонда по итерациям INLP, число удаленных направлений, точность классификатора тональности после удаления свойства и после удаления того же числа случайных направлений.**

In [ ]:
from sklearn.linear_model import LogisticRegression

LAYER = 5
P = None                                  # проекция; None — без вмешательства


def project(module, inputs, output):
    if P is None:
        return output
    return output @ P                       # в transformers 5 блок DistilBERT возвращает тензор


hook = clf.distilbert.transformer.layer[LAYER - 1].register_forward_hook(project)


@torch.no_grad()
def reps_and_preds(sentences, bs=64):
    reps, preds = [], []
    for i in range(0, len(sentences), bs):
        enc = tok(sentences[i:i + bs], return_tensors='pt', padding=True).to(device)
        out = clf(**enc, output_hidden_states=True)
        m = enc['attention_mask'][..., None]
        reps.append(((out.hidden_states[LAYER] * m).sum(1) / m.sum(1)).cpu())
        preds.append(out.logits.argmax(-1).cpu())
    return torch.cat(reps).numpy(), torch.cat(preds).numpy()


train = load_dataset('stanfordnlp/sst2', split='train').shuffle(seed=SEED)
train = train.filter(lambda r: len(r['sentence'].split()) >= 6).select(range(4000))
X_tr, _ = reps_and_preds(train['sentence'])
X_va, pred_va = reps_and_preds(sst['sentence'])
y_va = np.array(sst['label'])
med = np.median([len(s.split()) for s in sst['sentence']])
props = {
    'sentiment': (np.array(train['label']), y_va),
    'length': (np.array([len(s.split()) > med for s in train['sentence']], int),
               np.array([len(s.split()) > med for s in sst['sentence']], int)),
}
print(f'точность без вмешательства: {100 * (pred_va == y_va).mean():.2f}%; медиана длины: {med:.0f} слов')

In [ ]:
# Ваш код здесь

Два свойства дают две разные картины, как у Elazar и соавторов. Длину предложения зонд извлекает почти без ошибок, но после ее удаления точность классификатора тональности остается на уровне случайного контроля: свойство в представлении есть, а модель для своей задачи им не пользуется. Тональность INLP за 40 итераций до уровня самого частого класса не доводит — на слое 5 она записана во многих направлениях, — но уже удаление этих 40 направлений снижает точность классификатора до 69.15%, а удаление 40 случайных — только до 90.71%: этим свойством модель пользуется.

Без контроля Rand второй вывод был бы неверным: падение точности после любой проекции могло бы объясняться одной лишь потерей 40 измерений.

## 6. Атака на SAE-представление заменой одного токена

Li и соавторы меняют один токен запроса так, чтобы SAE-представление ушло от исходного, и показывают, что концепт в активациях самой LLM при этом сохраняется. Здесь та же схема на GPT-2 small и SAE `gpt2-small-res-jb` на `blocks.7.hook_resid_pre` — та пара модели и SAE, что в тетради `PRACTICE_sae_lens`. Процедура повторяет код авторов (`src/attacks/replacement.py` в AI4LIFE-GROUP/sae_robustness) с их значениями по умолчанию:

- **Вход** — запрос, к которому дописана инструкция `"\nThe previous text is about"`; представление берут на последнем токене. В коде авторов инструкция именно такая, в статье — «The previous sentence is about».
- **Представление и сходство** — множество $k$ самых активных латентов SAE и concept overlap ratio, доля общих индексов двух множеств. $k$ — число ненулевых латентов исходного запроса, как в статье.
- **Нецелевая атака на группу латентов** — для каждой позиции $t$ запроса (кроме первого токена) отдельная атака, которая меняет только токен $t$. Это 20 шагов GCG: градиент $-\cos(z,z_0)$ по one-hot вектору токенов дает 300 токенов-кандидатов для позиции $t$, из них случайно берут 100 и точно проверяют, лучший по overlap становится новым запросом. Итог — средний по позициям лучший overlap.
- **Точка сравнения** — те же 20 шагов по 100 случайных токенов на позицию, как режим `--random` у авторов.
- **Целевая индивидуальная атака** — 5 латентов несвязанного запроса «The stock market fell sharply after the announcement», активных у него и не входящих в множество нашего запроса. Для каждой позиции и каждого латента — до 10 шагов GCG по $\log\mathrm{softmax}(z)$ этого латента; латент включен, если его активация стала больше нуля. Итог — средняя по позициям доля включенных латентов.
- **Что сохраняется** — у авторов это проверяет линейный зонд на активациях LLM; здесь проще: косинус остаточного потока на последнем токене до и после лучшей нецелевой замены.

In [ ]:
from sae_lens import SAE, HookedSAETransformer

hook.remove()                             # проекция INLP больше не нужна
lm = HookedSAETransformer.from_pretrained('gpt2-small', device=device)
sae = SAE.from_pretrained(release='gpt2-small-res-jb', sae_id='blocks.7.hook_resid_pre', device=device)
HOOK = sae.cfg.metadata.hook_name
W_E = lm.W_E
EOS = lm.tokenizer.eos_token_id
SUFFIX = '\nThe previous text is about'   # инструкция из кода авторов
M_TOP, BATCH, ITERS_POP, ITERS_IND, N_LAT = 300, 100, 20, 10, 5   # значения по умолчанию из src/config.py авторов
gen = torch.Generator().manual_seed(SEED)


def last_state(toks):
    with torch.no_grad():
        _, cache = lm.run_with_cache(toks, names_filter=HOOK)
        h = cache[HOOK][:, -1]
        return h, sae.encode(h)


def overlap(s_batch, s_ref):
    ref = set(s_ref.tolist())
    return np.array([len(set(row.tolist()) & ref) / len(ref) for row in s_batch])


def token_grad(toks, objective):
    # градиент цели по one-hot матрице токенов; токен конца текста не предлагаем, как у авторов
    onehot = torch.zeros(toks.shape[1], W_E.shape[0], device=device)
    onehot[torch.arange(toks.shape[1]), toks[0]] = 1
    onehot.requires_grad_(True)
    store = {}
    lm.run_with_hooks(toks, fwd_hooks=[('hook_embed', lambda x, hook: (onehot @ W_E)[None]),
                                       (HOOK, lambda x, hook: store.update(h=x))])
    objective(sae.encode(store['h'][0, -1])).backward()
    g = onehot.grad
    g[:, EOS] = -float('inf')
    return g


def candidates(x, t, objective, random):
    if random:
        return torch.randint(0, W_E.shape[0], (BATCH,), generator=gen).to(device)
    top_m = torch.topk(token_grad(x, objective)[t], M_TOP).indices
    return top_m[torch.randint(0, M_TOP, (BATCH,), generator=gen).to(device)]


def population_attack(toks, n_x1, z0, s0, random=False):
    """Нецелевая атака авторов: для каждой позиции t запроса, кроме первой, ITERS_POP жадных шагов, меняющих только токен t. Возвращает лучший overlap по позициям и лучшие запросы."""
    k, best, best_x = len(s0), [], []
    for t in range(1, n_x1):
        x, b, bx = toks.clone(), 1.0, toks
        for _ in range(ITERS_POP):
            batch = x.repeat(BATCH, 1)
            batch[:, t] = candidates(x, t, lambda z: -torch.nn.functional.cosine_similarity(z, z0, dim=0), random)
            _, zb = last_state(batch)
            ov = overlap(torch.topk(zb, k).indices, s0)
            j = int(ov.argmin())
            x = batch[j:j + 1]
            if ov[j] < b:
                b, bx = ov[j], x
        best.append(b)
        best_x.append(bx)
    return np.array(best), best_x


def individual_attack(toks, n_x1, latents):
    """Целевая индивидуальная атака авторов: для каждой позиции и каждого латента до ITERS_IND шагов GCG; латент включен, если активация > 0. Возвращает среднюю по позициям долю включенных латентов."""
    rates = []
    for t in range(1, n_x1):
        on = 0
        for f in latents:
            x = toks.clone()
            for _ in range(ITERS_IND):
                batch = x.repeat(BATCH, 1)
                batch[:, t] = candidates(x, t, lambda z: torch.log_softmax(z, 0)[f], False)
                _, zb = last_state(batch)
                rank = (zb > zb[:, f:f + 1]).sum(1)          # ранг латента: сколько латентов активнее его
                j = int(rank.argmin())
                x = batch[j:j + 1]
                if zb[j, f] > 0:
                    on += 1
                    break
        rates.append(on / len(latents))
    return np.mean(rates)


In [ ]:
PROMPTS = ['How do I bake a chocolate cake at home?', 'The capital of France is a city called',
           'My favourite football team won the match yesterday', 'Write a short poem about the ocean at night',
           'The doctor said that the patient needs more rest', 'Please explain how a computer stores numbers']
UNRELATED = 'The stock market fell sharply after the announcement'
_, z_u = last_state(lm.to_tokens(UNRELATED + SUFFIX))
s_u = torch.topk(z_u[0], int((z_u[0] > 0).sum())).indices          # латенты несвязанного запроса, активные у него
rows_sae = []
for p in PROMPTS:
    n_x1 = lm.to_tokens(p).shape[1]
    toks = lm.to_tokens(p + SUFFIX)
    h0, z0 = last_state(toks)
    s0 = torch.topk(z0[0], int((z0[0] > 0).sum())).indices          # k — число ненулевых латентов исходного запроса
    ov_g, x_g = population_attack(toks, n_x1, z0[0], s0)
    ov_r, _ = population_attack(toks, n_x1, z0[0], s0, random=True)
    j = int(ov_g.argmin())
    h_g, _ = last_state(x_g[j])
    cos_g = torch.nn.functional.cosine_similarity(h_g, h0).item()
    cand = s_u[~torch.isin(s_u, s0)]
    latents = cand[torch.topk(z_u[0, cand], min(N_LAT, len(cand))).indices].tolist()   # 5 самых активных у несвязанного запроса латентов вне множества нашего
    rate = individual_attack(toks, n_x1, latents)
    rows_sae.append(dict(prompt=p, k=len(s0), gcg=ov_g.mean(), rnd=ov_r.mean(), gcg_best=ov_g.min(), gcg_cos=cos_g, ind=rate))
    print(f'{p} | k = {len(s0)}')
    print(f'   нецелевая GCG: средний по позициям overlap {ov_g.mean():.3f}, лучший {ov_g.min():.3f}, косинус остаточного потока {cos_g:.3f} | случайные токены: {ov_r.mean():.3f}')
    print(f'   лучший атакованный запрос: {lm.to_string(x_g[j][0, 1:n_x1])!r} | целевая индивидуальная: включено {rate:.2f} латентов')


**Задание 4. По `rows_sae` посчитайте средние по шести запросам: overlap после нецелевой GCG-замены (среднее по позициям), тот же overlap для случайных токенов, косинус остаточного потока после лучшей GCG-замены и долю латентов, включенных целевой индивидуальной атакой.**

In [ ]:
# Ваш код здесь

Сравните средние задания 4. Сколько из $k\approx52$ активных латентов исходного запроса оставляет одна замена токена и насколько при этом меняется остаточный поток на последнем токене? Насколько GCG уводит множество дальше случайного перебора при том же числе проверок? Если разрыв между ними невелик, на этой паре модели и SAE множество активных латентов хрупкое само по себе.

Долю латентов, включенных целевой индивидуальной атакой, сравните с Li и соавторами: у них на Gemma2 и Llama3 эта доля больше 0.6. Выше она или ниже на GPT-2 small с SAE `gpt2-small-res-jb`, и сильно ли при этом меняется остаточный поток?

## Что унести из тетради

- **Атрибуцию можно сдвинуть, не меняя класса.** Замены синонимами, выбранные по расстоянию атрибуций, снижают корреляцию с исходной IG-атрибуцией сильнее, чем случайные синонимы, при сохраненном предсказании.
- **Сначала мерить, насколько объяснение меняется без противника.** Смена опорного входа IG меняет ранги слов сопоставимо с атакой; число шагов IG почти не меняет.
- **Кандидаты определяют силу атаки.** С синонимами WordNet кандидатов меньше, чем с counter-fitted эмбеддингами статей, и разрыв между атакой и случайной точкой сравнения меньше, чем в статьях.
- **Зонд нашел — не значит, что модель использует.** Длина извлекается почти без ошибок, но ее удаление не меняет поведения классификатора; удаление тональности меняет. Различить это позволяет только контроль случайными направлениями.
- **SAE-представление хрупче активаций, из которых его строят.** Одна замена токена меняет множество активных латентов SAE сильнее, чем случайный перебор, а остаточный поток при этом меняется умеренно; целевая атака одним токеном включает часть выбранных латентов несвязанного запроса. Числа — в выводе задания 4.